# DeepSeek V3.2 模型结构完整实现

## 一、完整模型代码（ModelArgs / 各核心模块 / Transformer）

In [ ]:
import math  # 导入数学库，用于log/sqrt等运算（RoPE频率、注意力缩放等）
from dataclasses import dataclass  # 导入dataclass装饰器，用于简洁地定义模型超参数数据类
from typing import Tuple, Optional, Literal  # 导入类型注解：元组、可选值、字面量类型

import torch  # 导入PyTorch核心库
from torch import nn  # 导入神经网络模块，提供Module/Parameter等基础组件
import torch.nn.functional as F  # 导入函数式API，提供softmax/embedding/layer_norm等无状态函数
import torch.distributed as dist  # 导入分布式通信模块，用于多GPU张量并行

from kernel import act_quant, fp8_gemm, fp8_index  # 从本地kernel模块导入自定义FP8量化和矩阵乘函数


# 全局分布式状态，默认单机单卡
world_size = 1   # 参与分布式推理的总进程数（GPU数量），默认1
rank = 0         # 当前进程的全局编号，默认0（主进程）
block_size = 128  # FP8量化的分组大小：每128个元素共享一个缩放因子

@dataclass
class ModelArgs:
    """
    模型超参数数据类，使用 @dataclass 装饰器定义，集中管理所有模型配置。

    属性说明:
        max_batch_size (int): 最大批量大小，决定 KV 缓存预分配的 batch 维度上界。
        max_seq_len (int): 最大序列长度，决定 KV 缓存预分配的 seq 维度上界。
        dtype (Literal["bf16", "fp8"]): 模型权重的计算精度，bf16 或 fp8 量化。
        scale_fmt (Optional[str]): FP8 缩放因子的存储格式，None 表示连续（dense）缩放。
        vocab_size (int): 词汇表大小，决定 embedding 矩阵和 LM head 的词汇维度。
        dim (int): 模型隐藏层维度（d_model），各模块的输入/输出特征数。
        inter_dim (int): 密集 FFN（MLP）的中间层维度，通常大于 dim（升维）。
        moe_inter_dim (int): MoE 专家 FFN 的中间层维度，每个专家独立使用该维度。
        n_layers (int): Transformer 的总层数（Block 数量）。
        n_dense_layers (int): 使用密集 MLP 的前 n 层数量，其余层使用 MoE。
        n_heads (int): 多头注意力的头数。
        n_routed_experts (int): 路由专家总数（MoE 路由层中可供选择的专家数）。
        n_shared_experts (int): 共享专家数，每个 token 都会经过这些专家（不参与路由）。
        n_activated_experts (int): 每个 token 经过路由后实际激活的路由专家数。
        n_expert_groups (int): 专家分组数，用于分组路由策略，1 表示不分组。
        n_limited_groups (int): 每次激活的最大专家组数，限制跨组路由。
        score_func (Literal["softmax", "sigmoid"]): MoE 路由的得分函数类型。
        route_scale (float): 路由权重的全局缩放系数。
        q_lora_rank (int): Query 低秩分解的秩，0 表示不使用低秩压缩。
        kv_lora_rank (int): KV 低秩分解的秩（MLA 核心压缩参数）。
        qk_nope_head_dim (int): QK 中不使用位置编码（NoPE）部分的每头维度。
        qk_rope_head_dim (int): QK 中使用旋转位置编码（RoPE）部分的每头维度。
        v_head_dim (int): Value 的每头维度。
        original_seq_len (int): 预训练时的原始最大序列长度（YaRN 外推基准）。
        rope_theta (float): RoPE 的基础频率 theta 参数。
        rope_factor (float): YaRN 长度外推的缩放因子。
        beta_fast (int): YaRN 快速插值的 beta 参数（高频边界圈数）。
        beta_slow (int): YaRN 慢速插值的 beta 参数（低频边界圈数）。
        mscale (float): YaRN 注意力缩放修正系数，补偿长序列下熵的变化。
        index_head_dim (int): Indexer 模块每个头的维度。
        index_topk (int): Indexer 每次选择的 top-k 候选 key 数量。
    """
    max_batch_size: int = 8             # 最大批量大小，决定KV缓存预分配的第一维
    max_seq_len: int = 4096 * 4         # 最大序列长度（16384），支持长上下文
    dtype: Literal["bf16", "fp8"] = "bf16"  # 权重精度：bf16或fp8量化
    scale_fmt: Optional[str] = None     # FP8缩放因子格式，None表示连续缩放
    vocab_size: int = 102400            # 词汇表大小
    dim: int = 2048                     # 模型隐藏层维度（d_model）
    inter_dim: int = 10944              # 密集FFN中间层维度
    moe_inter_dim: int = 1408           # MoE专家FFN中间层维度（比inter_dim小）
    n_layers: int = 27                  # Transformer总层数
    n_dense_layers: int = 1             # 使用密集MLP的前n层数量
    n_heads: int = 16                   # 注意力头数
    # --- MoE 相关参数 ---
    n_routed_experts: int = 16          # 路由专家总数（此处改为16以降低显存需求；原版为64）
    n_shared_experts: int = 2           # 共享专家数（每次都激活）
    n_activated_experts: int = 6        # 每token激活的路由专家数
    n_expert_groups: int = 1            # 专家分组数，1表示不分组
    n_limited_groups: int = 1           # 每次激活的最大专家组数
    score_func: Literal["softmax", "sigmoid"] = "softmax"  # MoE路由得分函数
    route_scale: float = 1.             # 路由权重缩放系数
    # --- MLA 相关参数 ---
    q_lora_rank: int = 0                # Query低秩秩，0表示不使用低秩分解
    kv_lora_rank: int = 512             # KV低秩秩（MLA核心压缩参数）
    qk_nope_head_dim: int = 128         # QK非位置编码部分每头维度
    qk_rope_head_dim: int = 64          # QK位置编码（RoPE）部分每头维度
    v_head_dim: int = 128               # Value每头维度
    # --- YaRN 位置外推参数 ---
    original_seq_len: int = 4096        # 预训练时的原始最大序列长度
    rope_theta: float = 10000.0         # RoPE基础频率theta
    rope_factor: float = 40             # YaRN长度外推缩放因子
    beta_fast: int = 32                 # YaRN快速插值beta参数（高频边界）
    beta_slow: int = 1                  # YaRN慢速插值beta参数（低频边界）
    mscale: float = 1.                  # YaRN注意力缩放修正系数
    # --- Indexer 相关参数 ---
    index_n_heads: int = 64             # Indexer模块的注意力头数
    index_head_dim: int = 128           # Indexer每个头的维度
    index_topk: int = 2048              # Indexer每次选择的top-k候选key数量

class ParallelEmbedding(nn.Module):
    """
    支持分布式张量并行的词嵌入层。

    将词汇表按进程数均匀切分，每个进程只持有一部分词汇的嵌入向量。
    通过 all_reduce 操作聚合所有进程的查找结果，得到完整嵌入。

    参数:
        vocab_size (int): 词汇表总大小，必须能被 world_size 整除。
        dim (int): 嵌入向量的维度（即 d_model）。
    """
    def __init__(self, vocab_size: int, dim: int):
        """
        初始化并行嵌入层，按进程数切分词汇表。

        参数:
            vocab_size (int): 词汇表总大小，必须能被 world_size 整除。
            dim (int): 嵌入向量维度（即 d_model）。

        内部状态:
            self.part_vocab_size (int): 本进程负责的词汇表分片大小 = vocab_size // world_size。
            self.vocab_start_idx (int): 本进程词汇表的起始 token id（包含）。
            self.vocab_end_idx (int): 本进程词汇表的结束 token id（不包含）。
            self.weight (nn.Parameter): 本地嵌入矩阵，形状 (part_vocab_size, dim)。
        """
        super().__init__()  # 调用父类nn.Module初始化
        self.vocab_size = vocab_size  # 保存词汇表总大小
        self.dim = dim  # 保存嵌入向量维度
        assert vocab_size % world_size == 0, f"Vocabulary size must be divisible by world size (world_size={world_size})"  # 断言词汇表大小能被进程数整除
        self.part_vocab_size = (vocab_size // world_size)  # 当前进程负责的词汇表分片大小
        self.vocab_start_idx = rank * self.part_vocab_size  # 当前进程的词汇表起始索引（闭区间）
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size  # 当前进程的词汇表结束索引（开区间）
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))  # 本地嵌入矩阵参数，形状(part_vocab_size, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        并行嵌入层前向传播。

        参数:
            x (torch.Tensor): 输入 token 索引张量，形状 (batch_size, seq_len)，
                              值范围 [0, vocab_size)。

        返回:
            torch.Tensor: 嵌入向量，形状 (batch_size, seq_len, dim)。
        """
        if world_size > 1:  # 多GPU模式：只查找本进程负责的词汇表范围
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)  # 标记不属于本进程词汇表范围的token，True=不属于本进程
            x = x - self.vocab_start_idx  # 将token id转为本进程的局部偏移量
            x[mask] = 0  # 将越界token id置0（查找后会得到0向量，后续会清零）
        y = F.embedding(x, self.weight)  # 查找本地嵌入矩阵，形状(batch, seq, dim)
        if world_size > 1:  # 多GPU模式：聚合各进程的查找结果
            y[mask] = 0  # 将不属于本进程的位置清零（防止引入随机初始化值）
            dist.all_reduce(y)  # all_reduce求和，每个位置只有一个进程有非零值
        return y  # 返回嵌入向量，形状(batch, seq, dim)


def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None,
           scale_fmt: Optional[str] = None) -> torch.Tensor:
    """
    执行线性变换 y = x @ weight^T（+ bias），自动适配 BF16 和 FP8 两种计算路径。

    参数:
        x (torch.Tensor): 输入张量，形状 (..., in_features)。
        weight (torch.Tensor): 权重张量，形状 (out_features, in_features)；
                               若为 FP8 格式（element_size=1），会走量化计算路径。
        bias (Optional[torch.Tensor]): 偏置张量，形状 (out_features,)，当前版本不支持，必须为 None。
        scale_fmt (Optional[str]): FP8 缩放因子的存储格式，None 表示连续缩放。

    返回:
        torch.Tensor: 线性变换结果，形状 (..., out_features)。

    说明:
        - 非 FP8 权重（BF16/FP32）：直接调用 F.linear 标准实现。
        - FP8 权重：对输入 x 量化后调用 fp8_gemm 自定义 CUDA 核函数。
    """
    assert bias is None  # 当前版本不支持偏置（FP8 GEMM 接口暂不支持 bias 融合）

    if weight.dtype != torch.float8_e4m3fn:
        # ── BF16 / FP32 权重路径（标准路径）──────────────────────────────────────
        # F.linear(x, weight): 执行 y = x @ weight^T
        # x: (..., in_features)，weight: (out_features, in_features)
        # 输出: (..., out_features)，dtype 与 x 一致
        return F.linear(x, weight)  # (..., out_features)

    else:
        # ── FP8 量化权重路径（加速路径）──────────────────────────────────────────
        # 核心思想：激活值 x 也量化为 FP8，与 FP8 权重一起使用自定义 CUDA 核函数计算，
        # 避免将 FP8 权重反量化回 BF16 再计算（节省显存带宽，加速推理）。

        # Step 1：对激活值 x 进行 block-wise FP8 量化
        # act_quant 输入: x (..., in_features)
        # act_quant 输出:
        #   x_fp8:  (..., in_features)，dtype float8_e4m3fn，量化后的激活值
        #   scale:  (..., in_features // block_size)，dtype float32，每 128 元素一个缩放因子
        x, scale = act_quant(x, block_size, scale_fmt)  # x: FP8 激活，scale: 激活缩放因子

        # Step 2：调用自定义 FP8 GEMM 核函数
        # fp8_gemm 接受：量化激活 x_fp8、激活缩放因子 scale、FP8 权重、权重缩放因子
        # weight 形状: (out_features, in_features)，dtype float8_e4m3fn
        # weight.scale 形状: (out_features // block_size, in_features // block_size)，dtype float32
        # 输出: (..., out_features)，dtype bfloat16（GEMM 内部反量化并以 BF16 输出）
        return fp8_gemm(x, scale, weight, weight.scale)  # (..., out_features)


class Linear(nn.Module):
    """
    自定义线性层，支持 BF16 和 FP8 量化权重，可选偏置。

    通过类变量 dtype 和 scale_fmt 统一控制全局权重精度，
    由 Transformer.__init__ 根据 ModelArgs 配置设置。

    参数:
        in_features (int): 输入特征数（线性层的输入维度）。
        out_features (int): 输出特征数（线性层的输出维度）。
        bias (bool): 是否添加偏置项，默认 False（当前 FP8 路径不支持偏置）。
        dtype (optional): 权重的数据类型，默认使用类变量 Linear.dtype（BF16 或 FP8）。
    """
    dtype = torch.bfloat16      # 类变量：全局默认权重类型，由Transformer.__init__设置
    scale_fmt: Optional[str] = None  # 类变量：FP8缩放因子格式，None表示连续缩放

    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        """
        初始化线性层，创建权重（及可选的 FP8 缩放因子、偏置）参数。

        参数:
            in_features (int): 输入特征维度。
            out_features (int): 输出特征维度。
            bias (bool): 是否创建偏置参数，默认 False。
            dtype: 权重的数据类型；为 None 时使用类变量 Linear.dtype（BF16 或 FP8）。

        内部状态:
            self.weight (nn.Parameter): 权重矩阵，形状 (out_features, in_features)。
            self.scale (nn.Parameter | None): FP8 权重的 block-wise 缩放因子，
                形状 (out_features//128, in_features//128)；BF16 时为 None。
            self.bias (nn.Parameter | None): 偏置向量，形状 (out_features,)；不启用时为 None。
        """
        super().__init__()  # 调用父类nn.Module初始化
        self.in_features = in_features    # 保存输入特征维度
        self.out_features = out_features  # 保存输出特征维度
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype or Linear.dtype))  # 初始化权重参数，形状(out, in)；dtype未指定时使用类变量
        if self.weight.element_size() == 1:  # 如果权重是FP8（element_size=1字节）
            scale_out_features = (out_features + block_size - 1) // block_size  # 输出维度方向的缩放因子数量（向上取整）
            scale_in_features = (in_features + block_size - 1) // block_size    # 输入维度方向的缩放因子数量（向上取整）
            self.weight.scale = self.scale = nn.Parameter(torch.empty(scale_out_features, scale_in_features, dtype=torch.float32))  # 为FP8权重创建block-wise缩放因子参数，形状(out//128, in//128)
        else:  # 非FP8权重不需要缩放因子
            self.register_parameter("scale", None)  # 注册为None参数（保持接口一致）
        if bias:  # 如果需要偏置
            self.bias = nn.Parameter(torch.empty(out_features))  # 初始化偏置，形状(out_features,)
        else:  # 不需要偏置
            self.register_parameter("bias", None)  # 注册为None参数

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        线性层前向传播，委托给通用 linear 函数处理 BF16/FP8 两种计算路径。

        参数:
            x (torch.Tensor): 输入张量，形状 (..., in_features)。

        返回:
            torch.Tensor: 线性变换结果，形状 (..., out_features)。
        """
        return linear(x, self.weight, self.bias, self.scale_fmt)  # 调用通用linear函数执行实际计算


class ColumnParallelLinear(Linear):
    """
    列并行线性层：将输出特征维度（out_features）按进程数均匀切分。

    每个进程只持有 out_features // world_size 个输出神经元对应的权重。
    常用于 MLP 的 w1/w3（升维）、MHA 的 QKV 投影等。

    参数:
        in_features (int): 输入特征数（全局，所有进程相同）。
        out_features (int): 输出特征数（全局总量），必须能被 world_size 整除。
        bias (bool): 是否添加偏置项，默认 False。
        dtype (optional): 权重数据类型，默认使用 Linear.dtype（BF16 或 FP8）。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        """
        初始化列并行线性层，按进程数切分输出维度。

        参数:
            in_features (int): 输入特征维度（全局，所有进程相同）。
            out_features (int): 全局输出特征维度，必须能被 world_size 整除。
            bias (bool): 是否创建偏置参数，默认 False。
            dtype: 权重数据类型；为 None 时使用类变量 Linear.dtype。

        内部状态:
            self.part_out_features (int): 本进程负责的输出维度 = out_features // world_size。
            self.weight (nn.Parameter): 本地权重，形状 (part_out_features, in_features)。
        """
        assert out_features % world_size == 0, f"Output features must be divisible by world size (world_size={world_size})"  # 断言输出维度可被进程数整除
        self.part_out_features = out_features // world_size  # 计算本进程负责的输出特征数
        super().__init__(in_features, self.part_out_features, bias, dtype)  # 只创建本进程部分的权重

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        列并行线性层前向传播，计算本进程负责的输出列分片。

        参数:
            x (torch.Tensor): 输入张量，形状 (..., in_features)。

        返回:
            torch.Tensor: 本进程的列分片输出，形状 (..., part_out_features)；
                          完整输出需 all_gather 后拼接。
        """
        y = linear(x, self.weight, self.bias, self.scale_fmt)  # 计算本进程负责的列部分输出，形状(..., part_out_features)
        return y  # 返回本地输出（完整输出需all_gather）


class RowParallelLinear(Linear):
    """
    行并行线性层：将输入特征维度（in_features）按进程数均匀切分。

    每个进程只持有 in_features // world_size 个输入维度对应的权重行。
    通过 all_reduce 求和得到完整输出，常用于 MLP 的 w2（降维）、注意力输出投影等。

    参数:
        in_features (int): 输入特征数（全局总量），必须能被 world_size 整除。
        out_features (int): 输出特征数（全局，所有进程相同）。
        bias (bool): 是否添加偏置项，默认 False。
        dtype (optional): 权重数据类型，默认使用 Linear.dtype（BF16 或 FP8）。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False,
                 reduce_output=True, dtype=None):
        """
        初始化行并行线性层，按进程数切分输入维度。

        参数:
            in_features (int): 全局输入特征维度，必须能被 world_size 整除。
            out_features (int): 输出特征维度（全局，所有进程相同）。
            bias (bool): 是否创建偏置参数，默认 False。
            reduce_output (bool): 是否在前向传播中对各进程结果执行 all_reduce 求和，
                                  默认 True；设为 False 时仅保留本地部分和（用于后续手动聚合）。
            dtype: 权重数据类型；为 None 时使用类变量 Linear.dtype。

        内部状态:
            self.part_in_features (int): 本进程负责的输入维度 = in_features // world_size。
            self.weight (nn.Parameter): 本地权重，形状 (out_features, part_in_features)。
        """
        assert in_features % world_size == 0, f"Input features must be divisible by world size (world_size={world_size})"  # 断言输入维度可被进程数整除
        self.part_in_features = in_features // world_size  # 计算本进程负责的输入特征数
        self.reduce_output = reduce_output  # 保存是否需要all_reduce的标志
        super().__init__(self.part_in_features, out_features, bias, dtype)  # 只创建本进程部分的权重

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        行并行线性层前向传播，计算本进程的部分内积后 all_reduce 求和。

        参数:
            x (torch.Tensor): 输入张量（本进程分片），形状 (..., part_in_features)。

        返回:
            torch.Tensor: 完整输出张量，形状 (..., out_features)，dtype 与输入一致。
        """
        y = linear(x, self.weight, None, self.scale_fmt)  # 计算本进程的部分内积（行并行分片）
        if self.reduce_output and world_size > 1:  # 需要聚合且是多GPU模式
            y = y.float()  # 转float32提高all_reduce精度
            dist.all_reduce(y)  # 对所有进程的部分内积求和，得到完整输出
        if self.bias is not None:  # 如果有偏置
            y += self.bias  # 加上偏置向量
        return y.type_as(x)  # 恢复为输入dtype并返回


class RMSNorm(nn.Module):
    """
    均方根层归一化（Root Mean Square Layer Normalization）。

    公式：RMSNorm(x) = x / sqrt(mean(x²) + eps) * weight
    相比 LayerNorm 去掉了均值中心化，计算更高效，在 LLM 中被广泛采用。
    支持可选的残差融合模式，减少内存读写次数。

    参数:
        dim (int): 输入特征的维度（归一化的最后一维大小）。
        eps (float): 数值稳定项，防止分母为零，默认 1e-6。
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        """
        初始化 RMSNorm 层，创建可学习缩放参数。

        参数:
            dim (int): 输入特征维度（归一化的最后一维大小）。
            eps (float): 数值稳定项，防止分母为零，默认 1e-6。

        内部状态:
            self.weight (nn.Parameter): 可学习缩放参数 gamma，
                形状 (dim,)，全 1 初始化，dtype 为 float32。
        """
        super().__init__()  # 调用父类初始化
        self.dim = dim    # 保存特征维度
        self.eps = eps    # 保存数值稳定项，防止除零
        self.weight = nn.Parameter(torch.ones(dim, dtype=torch.float32))  # 可学习缩放参数，全1初始化，使用float32提高归一化精度

    def forward(self, x: torch.Tensor, residual: Optional[torch.Tensor] = None):
        """
        RMSNorm 前向传播，支持可选的残差融合以减少内存读写次数。

        参数:
            x (torch.Tensor): 输入张量，形状 (..., dim)，dtype 任意（内部转 float32 计算）。
            residual (Optional[torch.Tensor]): 可选残差张量，形状与 x 相同；
                若提供，则先执行 x = x + residual 再做归一化。

        返回:
            当 residual 为 None 时：
                torch.Tensor，归一化结果，形状 (..., dim)，dtype 与输入一致。
            当 residual 不为 None 时：
                Tuple[torch.Tensor, torch.Tensor]，
                第一个元素为归一化结果，形状 (..., dim)；
                第二个元素为融合后的残差（已更新为 x + residual），形状 (..., dim)。
        """
        dtype = x.dtype  # 记录输入原始dtype，用于最后恢复类型
        if residual is None:  # 无残差模式：直接归一化
            x = x.float()  # 转float32进行高精度计算（BF16精度不足以保证归一化稳定）
            var = x.pow(2).mean(-1, keepdim=True)  # 计算最后一维的均方值（RMS的平方），形状(..., 1)
            x = x * torch.rsqrt(var + self.eps)  # 乘以RMS倒数完成归一化：x / sqrt(mean(x^2) + eps)
            return (self.weight * x).to(dtype)  # 乘以可学习缩放参数后恢复原始dtype
        else:  # 有残差模式：先融合残差再归一化
            x = residual = x.float() + residual.float()  # 计算残差融合：新x = 旧x + residual（均用float32），同时更新residual
            var = x.pow(2).mean(-1, keepdim=True)  # 计算融合后张量的均方值，形状(..., 1)
            x = x * torch.rsqrt(var + self.eps)  # 归一化操作
            return (self.weight * x).to(dtype), residual.to(dtype)  # 返回归一化结果和更新后的残差（均恢复原dtype）


class LayerNorm(nn.Module):
    """
    标准层归一化（Layer Normalization），用于Indexer模块的Key归一化。
    公式：LayerNorm(x) = (x - mean(x)) / std(x) * weight + bias
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        """
        初始化 LayerNorm 层，创建可学习的缩放（gamma）和偏移（beta）参数。

        参数:
            dim (int): 输入特征维度（归一化的最后一维大小）。
            eps (float): 数值稳定项，防止方差为零时除零，默认 1e-6。

        内部状态:
            self.weight (nn.Parameter): 可学习缩放参数 gamma，形状 (dim,)，全 1 初始化，dtype=float32。
            self.bias (nn.Parameter):   可学习偏移参数 beta，形状 (dim,)，全 0 初始化，dtype=float32。
        """
        super().__init__()  # 调用父类初始化
        self.dim = dim  # 特征维度
        self.eps = eps  # 数值稳定项
        self.weight = nn.Parameter(torch.ones(dim, dtype=torch.float32))   # 可学习缩放参数，全1初始化
        self.bias = nn.Parameter(torch.zeros(dim, dtype=torch.float32))    # 可学习偏移参数，全0初始化

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        LayerNorm 前向传播：先转 float32 高精度计算，再恢复为输入 dtype。

        公式：y = (x - mean(x)) / sqrt(var(x) + eps) * weight + bias

        参数:
            x (torch.Tensor): 输入张量，形状任意 (..., dim)，
                              在 Indexer 中通常为 (bsz, seqlen, head_dim)。

        返回:
            torch.Tensor: LayerNorm 归一化结果，形状与输入相同 (..., dim)，
                          dtype 恢复为输入的原始类型（如 bfloat16）。
        """
        return F.layer_norm(x.float(), (self.dim,), self.weight, self.bias, self.eps).type_as(x)
        # x.float()：转 float32 保证归一化精度（BF16 范围不足）
        # (self.dim,)：仅对最后一维做归一化
        # .type_as(x)：将 float32 结果还原为输入的原始 dtype，shape 不变


def precompute_freqs_cis(args: ModelArgs) -> torch.Tensor:
    """
    预计算 RoPE 旋转位置编码所需的复数频率因子（带 YaRN 长度外推支持）。

    实现逻辑：
        1. 计算标准 RoPE 基础频率：freqs[i] = 1 / theta^(2i/dim)
        2. 若序列长度超过预训练长度，启用 YaRN 插值：
           - 低频维度：频率除以 factor（拉伸位置区间）
           - 高频维度：频率保持不变（精确的局部位置信息）
           - 中间维度：线性平滑插值
        3. 外积展开为 (seqlen, dim//2) 的角度矩阵
        4. 转为复数形式 e^(j*freq)

    参数:
        args (ModelArgs): 模型超参数对象，使用 qk_rope_head_dim、max_seq_len、
                          original_seq_len、rope_theta、rope_factor、
                          beta_fast、beta_slow 等字段。

    返回:
        torch.Tensor: 预计算的 RoPE 复数因子，形状 (max_seq_len, qk_rope_head_dim//2)，
                      复数类型（torch.complex64），每个元素为 cos + j*sin。
    """
    dim = args.qk_rope_head_dim      # int：RoPE 频率总维度数，freqs 将有 dim//2 个频率分量
    seqlen = args.max_seq_len         # int：预计算的最大序列长度，freqs_cis 第 0 维大小
    beta_fast = args.beta_fast        # int：高频边界对应的旋转圈数（YaRN 参数，通常=32）
    beta_slow = args.beta_slow        # int：低频边界对应的旋转圈数（YaRN 参数，通常=1）
    base = args.rope_theta            # float：RoPE 基础频率 θ（通常=10000）
    factor = args.rope_factor         # float：YaRN 长度外推缩放因子（通常=40）

    def find_correction_dim(num_rotations, dim, base, max_seq_len):
        """
        计算给定旋转圈数对应的 RoPE 频率维度索引（YaRN 插值边界计算）。

        公式：dim * log(max_seq_len / (num_rotations * 2π)) / (2 * log(base))

        参数:
            num_rotations (float): 目标旋转圈数（beta_fast 或 beta_slow）。
            dim (int): RoPE 的总频率维度数（qk_rope_head_dim）。
            base (float): RoPE 基础频率 theta。
            max_seq_len (int): 预训练时的原始最大序列长度。

        返回:
            float: 对应的频率维度索引（连续值，需取整后使用）。
        """
        return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))

    def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
        """
        计算 YaRN 插值的频率维度边界范围（高频端和低频端的分界索引）。

        参数:
            low_rot (float): 高频边界对应的旋转圈数（beta_fast，较大值）。
            high_rot (float): 低频边界对应的旋转圈数（beta_slow，较小值）。
            dim (int): RoPE 的总频率维度数（qk_rope_head_dim）。
            base (float): RoPE 基础频率 theta。
            max_seq_len (int): 预训练时的原始最大序列长度。

        返回:
            Tuple[int, int]: 插值边界的整数维度索引对 (low, high)，
                             已 clamp 到有效范围 [0, dim-1]。
        """
        low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))
        high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))
        return max(low, 0), min(high, dim-1)

    def linear_ramp_factor(min, max, dim):
        """
        生成线性斜坡平滑权重，用于 YaRN 高频/低频区间之间的平滑过渡。

        在 [min, max] 区间内线性插值从 0 到 1，区间外 clamp 到 [0, 1]。
        返回值越接近 1 表示越倾向于保持原始频率（高频），越接近 0 表示越倾向于缩放（低频）。

        参数:
            min (float): 线性斜坡的起始维度索引（此处值对应 smooth=0）。
            max (float): 线性斜坡的结束维度索引（此处值对应 smooth=1）。
            dim (int): 生成的张量维度数（等于 qk_rope_head_dim // 2）。

        返回:
            torch.Tensor: 平滑权重张量，形状 (dim,)，值域 [0, 1]，float32 类型。
        """
        if min == max:
            max += 0.001  # 防止分母为零导致 NaN（两边界重合时微小偏移）
        linear_func = (torch.arange(dim, dtype=torch.float32) - min) / (max - min)
        # torch.arange(dim) 生成 [0,1,...,dim-1]，减 min 后除以区间宽度
        # 输出 shape (dim,)，dtype float32，值在 [min,max] 之间为线性斜坡，两端外侧超出[0,1]
        ramp_func = torch.clamp(linear_func, 0, 1)  # clamp 到 [0,1]，shape 不变 (dim,)
        return ramp_func  # 返回平滑权重，shape (dim,)，float32，值域 [0,1]

    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
    # 计算标准RoPE基础频率：freqs[i] = 1 / theta^(2i/dim)，形状(dim//2,)
    if seqlen > args.original_seq_len:  # 如果序列长度超过预训练长度，启用YaRN外推
        low, high = find_correction_range(beta_fast, beta_slow, dim, base, args.original_seq_len)
        # 返回两个整数：high 为低频边界索引，low 为高频边界索引（均在 [0, dim-1] 范围内）
        smooth = 1 - linear_ramp_factor(low, high, dim // 2)
        # linear_ramp_factor 返回 shape(dim//2,)；1-ramp 后：高频端 smooth≈1（保持原频率），低频端 smooth≈0（启用缩放）
        freqs = freqs / factor * (1 - smooth) + freqs * smooth
        # YaRN 混合公式：低频分量除以 factor（压缩波长以外推），高频分量保持不变
        # freqs shape 仍为 (dim//2,)，与 smooth 按元素广播相乘

    t = torch.arange(seqlen)  # 生成位置索引[0, 1, ..., seqlen-1]，形状(seqlen,)
    freqs = torch.outer(t, freqs)  # 外积：位置×频率，形状(seqlen, dim//2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # 转为复数：e^(j*freqs) = cos + j*sin，形状(seqlen, dim//2)
    return freqs_cis  # 返回预计算的RoPE复数因子


def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor, interleaved: bool = True) -> torch.Tensor:
    """
    将预计算的 RoPE 旋转位置编码应用到输入张量（支持交错和非交错两种布局）。

    参数:
        x (torch.Tensor): 需要施加位置编码的输入张量，最后一维为 rope_head_dim，
                          形状通常为 (batch, seq, n_heads, rope_head_dim)。
        freqs_cis (torch.Tensor): 预计算的 RoPE 复数因子，形状 (seq_len, rope_head_dim//2)，
                                  由 precompute_freqs_cis 生成。
        interleaved (bool): 是否为交错布局（偶数索引/奇数索引交替排列），默认 True。
                            MLA 主注意力使用 True，Indexer 使用 False。

    返回:
        torch.Tensor: 施加旋转位置编码后的张量，形状与输入相同，dtype 与输入一致。
    """
    dtype = x.dtype   # 记录输入原始 dtype，最后用 .to(dtype) 恢复（避免 float32 输出）
    shape = x.shape   # 记录输入原始 shape，例如 (bsz, seqlen, n_heads, rope_head_dim)
    if not interleaved:  # 非交错布局（Indexer 使用）：RoPE 分量排列为 [x0,x1,...,xN/2,xN/2+1,...,xN-1]
        x = x.view(*shape[:-1], 2, -1).transpose(-1, -2).contiguous()
        # view 后: (..., 2, rope_head_dim//2)；transpose(-1,-2) 后: (..., rope_head_dim//2, 2)
        # contiguous() 保证内存连续（下一步 view_as_complex 要求连续）
    x = torch.view_as_complex(x.float().view(*shape[:-1], -1, 2))
    # .float(): 转为 float32，避免 view_as_complex 对 bfloat16 不支持
    # .view(*shape[:-1], -1, 2): 把最后一维拆成 (rope_dim//2, 2)，即每对相邻元素视为复数实部/虚部
    # view_as_complex 后: shape (..., rope_dim//2)，dtype complex64
    freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))
    # 原始 freqs_cis: (seqlen, rope_dim//2)；reshape 后: (1, seqlen, 1, rope_dim//2)
    # 用于与 x: (bsz, seqlen, n_heads, rope_dim//2) 广播相乘
    y = torch.view_as_real(x * freqs_cis).flatten(3)
    # x * freqs_cis: 复数相乘实现旋转，shape 不变 (bsz, seqlen, n_heads, rope_dim//2)，dtype complex64
    # view_as_real 后: (bsz, seqlen, n_heads, rope_dim//2, 2)；flatten(3) 从第 3 维合并最后两维
    # 最终 y: (bsz, seqlen, n_heads, rope_dim)，dtype float32
    if not interleaved:  # 非交错布局：旋转后需将交错顺序恢复为原始非交错顺序
        y = torch.cat([y[..., 0::2], y[..., 1::2]], dim=-1)
        # 0::2 取偶数索引（原 x_0~x_{N/2-1}），1::2 取奇数索引（原 x_{N/2}~x_{N-1}）
        # cat 后恢复非交错排列，shape 不变 (bsz, seqlen, n_heads, rope_dim)
    return y.to(dtype)  # 恢复为输入原始 dtype（如 bfloat16），shape 不变


def rotate_activation(x: torch.Tensor) -> torch.Tensor:
    """
    对输入张量应用 Hadamard 变换（随机旋转激活），用于 LightningIndexer 的特征分散。

    核心作用：
        通过正交的 Hadamard 矩阵 H 对特征向量做旋转变换（H·x），
        使得原本集中在少数维度的信息均匀分散到所有维度。
        这样 FP8 量化时每个 block 的数值范围更均匀，量化误差更小。
        由于 H 是正交矩阵，旋转不改变向量的内积：Hq · Hk = q · k，
        因此不影响注意力得分的正确性。

    数学原理：
        对输入每个向量执行 H·x / sqrt(N)，其中 N = hidden_size，
        scale=N^(-0.5) 使变换后的向量方差保持不变（H 的每行 L2 范数为 sqrt(N)）。

    参数:
        x (torch.Tensor): 输入张量，dtype 必须为 bfloat16（fast_hadamard_transform 要求），
                          最后一维 hidden_size 必须为 2 的幂次，
                          形状任意，如 (bsz, seqlen, n_heads, head_dim) 或 (bsz, seqlen, head_dim)。

    返回:
        torch.Tensor: Hadamard 变换后的张量，shape 与输入完全相同，dtype 仍为 bfloat16。
    """
    assert x.dtype == torch.bfloat16, \
        f"rotate_activation 要求输入为 bfloat16，但得到 {x.dtype}"  # fast_hadamard_transform 仅支持 bfloat16
    from fast_hadamard_transform import hadamard_transform  # 延迟导入，避免无 GPU 环境时导入报错
    hidden_size = x.size(-1)  # int：最后一维大小，即被旋转的向量维度（必须为 2 的幂次，如 64/128）
    return hadamard_transform(x, scale=hidden_size ** -0.5)
    # 执行 H·x / sqrt(hidden_size)，shape 不变；scale 归一化保持向量方差为 1


class Indexer(torch.nn.Module):
    """
    索引器模块 - 用于在长序列注意力计算中进行稀疏化，减少计算量。

    核心思想：
        不是计算 query 和所有 key 的注意力，而是先通过一个轻量级的索引机制
        快速筛选出最相关的 top-k 个位置，然后只在这些位置上计算完整的注意力。

    工作流程：
        1. 使用低秩投影将 query 和 key 映射到较小的索引空间
        2. 在索引空间中快速计算相似度得分（FP8 量化加速）
        3. 选出 top-k 个最相关的位置索引
        4. MLA 只在这些索引位置计算完整注意力（稀疏化）

    参数:
        args (ModelArgs): 模型超参数对象，需包含 dim、index_n_heads、
                          index_head_dim、qk_rope_head_dim、index_topk、
                          q_lora_rank、scale_fmt、max_batch_size、max_seq_len。
    """
    def __init__(self, args: ModelArgs):
        super().__init__()  # 调用父类初始化
        self.dim: int = args.dim              # 模型隐藏层维度
        self.n_heads: int = args.index_n_heads  # 索引器注意力头数
        self.n_local_heads = args.index_n_heads // world_size  # 本进程负责的头数
        self.head_dim: int = args.index_head_dim    # 索引器每个头的维度
        self.rope_head_dim: int = args.qk_rope_head_dim  # RoPE编码维度（与MLA共享）
        self.index_topk: int = args.index_topk      # 每次选择的top-k候选数
        self.q_lora_rank: int = args.q_lora_rank    # Query低秩秩
        self.wq_b = Linear(self.q_lora_rank, self.n_heads * self.head_dim)  # Query投影：权重形状(n_heads*head_dim, q_lora_rank)，低秩→多头索引空间
        self.wk = Linear(self.dim, self.head_dim)  # Key投影：权重形状(head_dim, dim)，隐藏层→单共享索引key（所有头共用同一个key）
        self.k_norm = LayerNorm(self.head_dim)  # Key的LayerNorm归一化，输入/输出形状(*, head_dim)
        # weights_proj存储为bf16，这里用fp32方便float计算时的精度
        self.weights_proj = Linear(self.dim, self.n_heads, dtype=torch.float32)  # 头权重投影：权重形状(n_heads, dim)，为每个头生成一个标量权重
        self.softmax_scale = self.head_dim ** -0.5  # 注意力缩放因子 1/sqrt(head_dim)
        self.scale_fmt = args.scale_fmt  # FP8缩放因子格式

        self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.head_dim, dtype=torch.float8_e4m3fn), persistent=False)
        # FP8格式的key缓存，形状(max_batch, max_seq, head_dim)，节省显存
        self.register_buffer("k_scale_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.head_dim // block_size, dtype=torch.float32), persistent=False)
        # key量化缩放因子缓存，形状(max_batch, max_seq, head_dim//128)


    def forward(self, x: torch.Tensor, qr: torch.Tensor, start_pos: int,
                freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        前向传播 - 计算索引得分并返回 top-k 位置索引。

        执行步骤：
            Step 1  从低秩 query 投影到多头索引 query 空间，并应用 RoPE
            Step 2  计算共享 key，做 LayerNorm 并应用 RoPE
            Step 3  对 query/key 应用 Hadamard 变换（增强特征分散性）
            Step 4  将 query/key 量化为 FP8 格式
            Step 5  将当前步 FP8 key 及缩放因子写入 KV 缓存
            Step 6  计算每个注意力头的标量权重并组合多个缩放因子
            Step 7  调用 FP8 核函数快速计算 Q @ K^T 索引得分
            Step 8  应用因果掩码（可选）
            Step 9  选出 top-k 个得分最高的 key 位置索引

        参数:
            x         (torch.Tensor): 输入特征，形状 (batch_size, seq_len, dim)
            qr        (torch.Tensor): 低秩 query 表示，形状 (batch_size, seq_len, q_lora_rank)
            start_pos (int)         : 当前序列在完整序列中的起始位置（用于 KV 缓存索引）
            freqs_cis (torch.Tensor): 预计算的 RoPE 复数因子，形状 (seq_len, rope_dim//2)
            mask      (Optional[torch.Tensor]): 因果注意力掩码，形状 (seq_len, seq_len)，
                                                None 表示 decode 单步推理

        返回:
            topk_indices (torch.Tensor): 每个 query 位置选出的 top-k 个 key 位置索引，
                                         形状 (batch_size, seq_len, min(index_topk, end_pos))，
                                         注意：各头得分已在 fp8_index 内部加权求和，
                                         故无 n_heads 维，直接是全局共享的 top-k 索引。
        """
        bsz, seqlen, _ = x.size()  # 解包批量大小、序列长度、特征维度
        end_pos = start_pos + seqlen  # 计算当前序列在KV缓存中的结束位置
        q = self.wq_b(qr)  # 低秩query→多头索引query，形状(batch, seq, n_heads*head_dim)
        q = q.view(bsz, seqlen, self.n_heads, self.head_dim)  # 重塑为多头形式
        q_pe, q_nope = torch.split(q, [self.rope_head_dim, self.head_dim - self.rope_head_dim], dim=-1)
        # 分离query的RoPE部分和非RoPE部分
        # q_pe 形状(bsz, seqlen, n_heads, rope_head_dim)；q_nope 形状(bsz, seqlen, n_heads, head_dim-rope_head_dim)
        # 注意：索引器使用非交错RoPE（interleaved=False），与MLA主注意力不同
        q_pe = apply_rotary_emb(q_pe, freqs_cis, False)  # 对RoPE部分应用旋转位置编码（非交错），形状不变(bsz,seqlen,n_heads,rope_head_dim)
        q = torch.cat([q_pe, q_nope], dim=-1)  # 拼接回完整query，形状(bsz, seqlen, n_heads, head_dim)
        k = self.wk(x)    # 计算共享key（所有头共用同一key），形状(bsz, seqlen, head_dim)
        k = self.k_norm(k)  # 对key做LayerNorm归一化，形状不变(bsz, seqlen, head_dim)
        k_pe, k_nope = torch.split(k, [self.rope_head_dim, self.head_dim - self.rope_head_dim], dim=-1)
        # 分离key的RoPE部分和非RoPE部分
        # k_pe 形状(bsz, seqlen, rope_head_dim)；k_nope 形状(bsz, seqlen, head_dim-rope_head_dim)
        # 注意：unsqueeze(2)增加head维度以匹配apply_rotary_emb的4D输入格式，squeeze(2)再去掉
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis, False).squeeze(2)  # key的RoPE编码，形状(bsz, seqlen, rope_head_dim)
        k = torch.cat([k_pe, k_nope], dim=-1)  # 拼接回完整key，形状(bsz, seqlen, head_dim)
        q = rotate_activation(q)  # 对query应用Hadamard变换（增强特征分散性），形状不变(bsz, seqlen, n_heads, head_dim)
        k = rotate_activation(k)  # 对key应用Hadamard变换，形状不变(bsz, seqlen, head_dim)
        q_fp8, q_scale = act_quant(q, block_size, self.scale_fmt)
        # q_fp8 形状(bsz, seqlen, n_heads, head_dim)，dtype=float8_e4m3fn
        # q_scale 形状(bsz, seqlen, n_heads, head_dim//block_size)，dtype=float32，每128元素共享一个缩放因子
        k_fp8, k_scale = act_quant(k, block_size, self.scale_fmt)
        # k_fp8 形状(bsz, seqlen, head_dim)，dtype=float8_e4m3fn
        # k_scale 形状(bsz, seqlen, head_dim//block_size)，dtype=float32
        self.k_cache[:bsz, start_pos:end_pos] = k_fp8  # 将当前步FP8 key写入缓存，写入位置(bsz, start_pos:end_pos, head_dim)
        self.k_scale_cache[:bsz, start_pos:end_pos] = k_scale  # 将key缩放因子写入缓存，写入位置(bsz, start_pos:end_pos, head_dim//block_size)
        weights = self.weights_proj(x.float()) * self.n_heads ** -0.5  # 计算每头标量权重，形状(bsz, seqlen, n_heads)，乘以1/sqrt(n_heads)缩放
        weights = weights.unsqueeze(-1) * q_scale * self.softmax_scale
        # unsqueeze后(bsz, seqlen, n_heads, 1) × q_scale(bsz, seqlen, n_heads, head_dim//block_size) × softmax_scale(标量)
        # 最终weights形状(bsz, seqlen, n_heads, head_dim//block_size)，融合了头权重、量化缩放、注意力缩放三个因子
        index_score = fp8_index(  # 调用FP8索引得分内核，对所有头加权求和后得到每个位置的综合得分
            q_fp8.contiguous(),                            # FP8量化的query，形状(bsz, seqlen, n_heads, head_dim)
            weights,                                       # 融合缩放因子，形状(bsz, seqlen, n_heads, head_dim//block_size)
            self.k_cache[:bsz, :end_pos].contiguous(),    # 历史FP8 key缓存，形状(bsz, end_pos, head_dim)
            self.k_scale_cache[:bsz, :end_pos].contiguous()  # 历史key缩放因子，形状(bsz, end_pos, head_dim//block_size)
        )  # 返回索引得分，形状(bsz, seqlen, end_pos)，各头已加权求和
        if mask is not None:  # 如果有注意力掩码（prefill阶段）
            index_score += mask  # 加到得分上（causal mask中-inf位置不会被选入top-k）
        topk_indices = index_score.topk(min(self.index_topk, end_pos), dim=-1)[1]
        # 选出得分最高的top-k个key索引；[1]取索引值（[0]是得分值）
        # topk_indices 形状(bsz, seqlen, min(index_topk, end_pos))，dtype=int64
        topk_indices_ = topk_indices.clone()  # 克隆用于分布式验证
        dist.broadcast(topk_indices_, src=0)  # 从rank 0广播topk结果
        assert torch.all(topk_indices == topk_indices_), f"{topk_indices=} {topk_indices_=}"  # 验证所有进程结果一致
        return topk_indices  # 返回top-k位置索引


def weight_dequant(weight, scale):
    """
    将FP8量化权重反量化为默认浮点类型（通常BF16）。
    用于decode阶段预先反量化wkv_b权重，避免每步重复执行。

    参数:
        weight (torch.Tensor): FP8量化的2D权重矩阵，形状 (out_features, in_features)，dtype float8_e4m3fn。
        scale  (torch.Tensor): block-wise缩放因子，形状 (out_features//128, in_features//128)，dtype float32。

    返回:
        反量化后的权重，形状与weight相同
    """
    shape = weight.shape  # 记录原始形状(out_features, in_features)
    assert weight.dim() == 2  # 断言为2D矩阵
    weight = weight.view(
        shape[0] // block_size, block_size, shape[1] // block_size, block_size
    ).transpose(1, 2).contiguous().view(-1, block_size * block_size)
    # 按block重组：将权重分成(out//128, 128, in//128, 128)块，转置后展平每个block
    weight = (weight.float() * scale.view(-1, 1).float()
              ).to(torch.get_default_dtype()
                   ).view(shape[0] // block_size, shape[1] // block_size, block_size, block_size
                          ).transpose(1, 2).contiguous().view(shape)
    # 每个block乘以对应缩放因子，转为默认dtype，再重组回原始形状
    return weight  # 返回反量化后的权重，形状(out, in)


class MLA(nn.Module):
    """
    多头潜在注意力层（Multi-Head Latent Attention，MLA）。

    DeepSeek V3 的核心创新：通过低秩压缩大幅减少 KV 缓存显存占用。
    KV 缓存只保存压缩后的低秩潜在表示（kv_lora_rank 维），而非完整的 K/V 矩阵。
    同时集成 Indexer 稀疏注意力，通过 top-k 筛选减少 prefill 阶段的计算量。

    主要属性:
        dim (int): 模型隐藏层维度（输入/输出特征数）。
        n_heads (int): 全局注意力头数。
        n_local_heads (int): 本进程负责的注意力头数（张量并行切分后）。
        q_lora_rank (int): Query 低秩分解的秩。
        kv_lora_rank (int): KV 低秩分解的秩（MLA 缓存的核心维度）。
        qk_nope_head_dim (int): QK 中不使用位置编码（NoPE）部分的每头维度。
        qk_rope_head_dim (int): QK 中使用旋转位置编码（RoPE）部分的每头维度。
        qk_head_dim (int): QK 的完整每头维度（nope + rope）。
        v_head_dim (int): Value 的每头维度。
        softmax_scale (float): 注意力得分的缩放因子（含 YaRN mscale 修正）。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 MLA 层，创建所有低秩投影矩阵、归一化层及 KV 缓存。

        参数:
            args (ModelArgs): 模型超参数对象，使用以下字段：
                dim、n_heads、q_lora_rank、kv_lora_rank、
                qk_nope_head_dim、qk_rope_head_dim、v_head_dim、
                scale_fmt、max_batch_size、max_seq_len、
                original_seq_len、mscale（YaRN softmax 修正）。

        内部状态（KV 缓存相关）:
            self.kv_cache  (Tensor): 低秩潜在 KV 缓存，形状 (max_batch_size, max_seq_len, kv_lora_rank)。
            self.pe_cache  (Tensor): Key 的 RoPE 部分缓存，形状 (max_batch_size, max_seq_len, qk_rope_head_dim)。
        """
        super().__init__()  # 调用父类 nn.Module 初始化
        self.dim = args.dim                      # int：模型隐藏层维度（所有线性层的 in/out dim）
        self.n_heads = args.n_heads              # int：全局注意力头数
        self.n_local_heads = args.n_heads // world_size  # int：本进程的注意力头数（张量并行切分）
        self.q_lora_rank = args.q_lora_rank      # int：Query 低秩分解的秩（wq_a 的输出维度）
        self.kv_lora_rank = args.kv_lora_rank    # int：KV 低秩压缩维度（MLA 缓存的核心大小）
        self.qk_nope_head_dim = args.qk_nope_head_dim  # int：QK 非位置编码部分的每头维度
        self.qk_rope_head_dim = args.qk_rope_head_dim  # int：QK 旋转位置编码部分的每头维度
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim  # int：QK 完整每头维度 = nope + rope
        self.v_head_dim = args.v_head_dim        # int：Value 的每头维度

        self.wq_a = Linear(self.dim, self.q_lora_rank)  # Query 第一阶段：权重(q_lora_rank, dim)，dim → q_lora_rank（低秩降维）
        self.q_norm = RMSNorm(self.q_lora_rank)  # 低秩空间的 RMSNorm，输入/输出 shape (*, q_lora_rank)
        self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)
        # Query 第二阶段：全局权重(n_heads*qk_head_dim, q_lora_rank)，本进程权重(n_local_heads*qk_head_dim, q_lora_rank)
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)
        # KV 联合投影：权重(kv_lora_rank+rope_dim, dim)，输出前 kv_lora_rank 维为潜在 KV，后 rope_dim 维为 key 的位置编码
        self.kv_norm = RMSNorm(self.kv_lora_rank)  # KV 潜在表示的 RMSNorm，输入/输出 shape (*, kv_lora_rank)
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))
        # KV 第二阶段：全局权重(n_heads*(qk_nope+v_dim), kv_lora_rank)，输出按头拆分为 k_nope 和 v
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)
        # 输出投影：全局权重(dim, n_heads*v_head_dim)，聚合多头后通过 all_reduce 得到最终输出
        self.softmax_scale = self.qk_head_dim ** -0.5  # float：注意力缩放因子 1/sqrt(qk_head_dim)，防止点积过大
        self.scale_fmt = args.scale_fmt          # Optional[str]：FP8 量化缩放因子格式
        if args.max_seq_len > args.original_seq_len:  # 启用 YaRN 长度外推时修正注意力缩放
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0
            # float：YaRN 修正系数，log(rope_factor) 捕捉外推幅度，0.1*mscale 为衰减强度
            self.softmax_scale = self.softmax_scale * mscale * mscale
            # 应用 mscale^2（两次缩放：一次对 q，一次对 k，等价于点积乘 mscale^2）

        self.indexer = Indexer(args)  # 创建稀疏注意力索引器

        self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)
        # KV低秩表示缓存（MLA核心优化）：只缓存压缩后的低秩表示，形状(max_batch, max_seq, kv_lora_rank)
        self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)
        # 位置编码缓存：单独缓存不可被低秩压缩的RoPE部分，形状(max_batch, max_seq, rope_dim)
        self.dequant_wkv_b = None  # decode阶段预反量化的wkv_b权重缓存（首次decode时初始化）

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        MLA 层前向传播，自动区分 prefill（mask 不为 None）和 decode（mask 为 None）两种模式。

        Prefill 模式（mask 不为 None）：处理整个输入序列，使用完整注意力 + Indexer 稀疏化。
        Decode 模式（mask 为 None）：增量推理，在低秩空间直接计算 Q·K^T，避免还原完整 K 矩阵。

        参数:
            x (torch.Tensor): 输入特征张量，形状 (batch_size, seq_len, dim)。
            start_pos (int): 当前序列在完整序列中的起始位置，用于 KV 缓存读写索引。
            freqs_cis (torch.Tensor): 预计算的 RoPE 复数因子，形状 (seq_len, rope_dim//2)。
            mask (Optional[torch.Tensor]): 因果注意力掩码，形状 (seq_len, seq_len)；
                                           decode 单步推理时为 None。

        返回:
            torch.Tensor: 注意力输出，形状与输入相同 (batch_size, seq_len, dim)。
        """
        bsz, seqlen, _ = x.size()  # 解包：x shape (bsz, seqlen, dim)，_ 即 dim
        end_pos = start_pos + seqlen  # int：当前步骤的 KV 缓存写入上界（不含），即历史长度
        qr = self.q_norm(self.wq_a(x))
        # wq_a(x): (bsz, seqlen, dim) → (bsz, seqlen, q_lora_rank)
        # q_norm 后 shape 不变，为低秩空间的归一化 query 表示，同时供 Indexer 复用
        q = self.wq_b(qr)  # (bsz, seqlen, q_lora_rank) → (bsz, seqlen, n_local_heads * qk_head_dim)
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)  # 拆分头：(bsz, seqlen, n_local_heads, qk_head_dim)
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)
        # q_nope: (bsz, seqlen, n_local_heads, qk_nope_head_dim)，不施加位置编码的部分
        # q_pe:   (bsz, seqlen, n_local_heads, qk_rope_head_dim)，将施加 RoPE 的部分
        q_pe = apply_rotary_emb(q_pe, freqs_cis)  # shape 不变 (bsz, seqlen, n_local_heads, qk_rope_head_dim)，interleaved=True
        kv = self.wkv_a(x)  # (bsz, seqlen, dim) → (bsz, seqlen, kv_lora_rank + qk_rope_head_dim)
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        # kv:   (bsz, seqlen, kv_lora_rank)，将被压缩缓存的潜在 KV 表示
        # k_pe: (bsz, seqlen, qk_rope_head_dim)，key 的 RoPE 位置编码部分（无法被低秩压缩）
        kv = self.kv_norm(kv)  # RMSNorm 后 shape 不变 (bsz, seqlen, kv_lora_rank)
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)
        # unsqueeze(2): (bsz, seqlen, qk_rope_head_dim) → (bsz, seqlen, 1, qk_rope_head_dim)（伪造 head 维）
        # RoPE 后 shape 不变 (bsz, seqlen, 1, qk_rope_head_dim)，1 头广播给所有真实头
        # 在实际部署中使用 FP8 KV cache，这里通过量化-反量化模拟 FP8 精度损失
        kv_fp8, kv_scale = act_quant(kv, block_size, self.scale_fmt)
        # kv_fp8:  (bsz, seqlen, kv_lora_rank)，dtype float8_e4m3fn
        # kv_scale:(bsz, seqlen, kv_lora_rank // block_size)，dtype float32，每 128 元素一个缩放因子
        kv = (kv_fp8.view(-1, block_size).float() * kv_scale.view(-1, 1)).to(kv.dtype).view_as(kv)
        # view(-1, block_size): 展平前两维后按 block 分组，shape (bsz*seqlen*kv_lora_rank//block_size, block_size)
        # *kv_scale.view(-1,1): 每组乘以对应缩放因子（广播），shape 不变
        # .to(kv.dtype).view_as(kv): 恢复 dtype 和原始 shape (bsz, seqlen, kv_lora_rank)
        # 意义：量化后立即反量化，模拟 FP8 精度损失（训练/推理对齐）
        self.kv_cache[:bsz, start_pos:end_pos] = kv    # 写入缓存区域 (bsz, seqlen, kv_lora_rank)
        self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)
        # squeeze(2): (bsz, seqlen, 1, qk_rope_head_dim) → (bsz, seqlen, qk_rope_head_dim)，去除伪造头维度
        if mask is not None:    # ── Prefill 模式（处理整段输入序列）──
            q = torch.cat([q_nope, q_pe], dim=-1)  # 拼接 nope 和 pe，shape (bsz, seqlen, n_local_heads, qk_head_dim)
            kv = self.wkv_b(kv)  # (bsz, seqlen, kv_lora_rank) → (bsz, seqlen, n_local_heads*(qk_nope_head_dim+v_head_dim))
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)  # 拆头：(bsz, seqlen, n_local_heads, qk_nope+v_dim)
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)
            # k_nope: (bsz, seqlen, n_local_heads, qk_nope_head_dim)
            # v:      (bsz, seqlen, n_local_heads, v_head_dim)
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)
            # k_pe.expand: (bsz,seqlen,1,qk_rope_head_dim) → (bsz,seqlen,n_local_heads,qk_rope_head_dim)（广播到每个头）
            # k: (bsz, seqlen, n_local_heads, qk_head_dim)，完整的 key 张量
            scores = torch.einsum("bshd,bthd->bsht", q, k).mul_(self.softmax_scale)
            # einsum: Q(bsz,s,h,d) @ K^T(bsz,t,h,d) → (bsz, seqlen_q=s, n_local_heads=h, seqlen_k=t)
            # .mul_(softmax_scale): 原地缩放，shape 不变 (bsz, seqlen, n_local_heads, seqlen)

            # ── 稀疏注意力：使用 LightningIndexer 筛选 top-k 候选 key ──
            topk_indices = self.indexer(x, qr, start_pos, freqs_cis, mask)
            # 返回 shape (bsz, seqlen, min(index_topk, end_pos))，int64，各头综合得分后的全局 top-k 索引
            index_mask = torch.full((bsz, seqlen, seqlen), float("-inf"), device=x.device).scatter_(-1, topk_indices, 0)
            # full 初始化全为 -inf，shape (bsz, seqlen, seqlen)
            # scatter_(-1, topk_indices, 0)：将 top-k 位置填 0（允许注意力），其余保持 -inf（屏蔽）
            index_mask += mask  # 与 causal mask(seqlen, seqlen) 相加，广播到 (bsz, seqlen, seqlen)，两者都满足才为 0
            scores += index_mask.unsqueeze(2)
            # unsqueeze(2): (bsz, seqlen, seqlen) → (bsz, seqlen, 1, seqlen)，广播到 scores (bsz, seqlen, n_heads, seqlen)

            scores = scores.softmax(dim=-1)  # 沿最后一维（seqlen_k）归一化，shape 不变 (bsz, seqlen, n_local_heads, seqlen)
            x = torch.einsum("bsht,bthd->bshd", scores, v)
            # scores(bsz,s,h,t) @ v(bsz,t,h,d) → 加权聚合，输出 (bsz, seqlen, n_local_heads, v_head_dim)
        else:                   # ── Decode 模式（增量推理，seqlen=1）──
            if self.dequant_wkv_b is None and self.wkv_b.scale is not None:
                # 触发条件：首次执行 decode 步骤 且 wkv_b 权重为 FP8 量化格式
                self.dequant_wkv_b = weight_dequant(self.wkv_b.weight, self.wkv_b.scale)
                # 预反量化 wkv_b 权重到 BF16，避免每步 decode 重复反量化（一次性开销）
                # 输出 shape (n_local_heads*(qk_nope_head_dim+v_head_dim), kv_lora_rank)，dtype bfloat16
            wkv_b = self.wkv_b.weight if self.dequant_wkv_b is None else self.dequant_wkv_b
            # 选择使用的权重：BF16 时用原始权重，FP8 时用预反量化权重
            # shape (n_local_heads*(qk_nope_head_dim+v_head_dim), kv_lora_rank)
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)
            # 拆头：(n_local_heads, qk_nope_head_dim+v_head_dim, kv_lora_rank)，后续按头切片使用
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            # q_nope(bsz,s=1,h,d=qk_nope) @ wkv_b_k(h,d=qk_nope,c=kv_lora_rank) → (bsz,1,n_local_heads,kv_lora_rank)
            # 关键优化：将 q_nope 投影到 KV 低秩空间，decode 时无需还原完整 K 矩阵
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) +
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale
            # 第一项：q_nope_lora(bsz,1,h,c) @ kv_cache^T(bsz,t,c) → (bsz,1,h,end_pos)，nope 部分得分（低秩空间直接计算）
            # 第二项：q_pe(bsz,1,h,r) @ pe_cache^T(bsz,t,r) → (bsz,1,h,end_pos)，RoPE 部分得分
            # kv_cache: (bsz, end_pos, kv_lora_rank)；pe_cache: (bsz, end_pos, qk_rope_head_dim)
            # 两项相加并乘以 softmax_scale，scores shape: (bsz, 1, n_local_heads, end_pos)

            # ── 稀疏注意力：使用 LightningIndexer 筛选 top-k 候选 key ──
            topk_indices = self.indexer(x, qr, start_pos, freqs_cis, mask)
            # decode 时 seqlen=1，返回 shape (bsz, 1, min(index_topk, end_pos))，int64
            index_mask = torch.full((bsz, 1, end_pos), float("-inf"), device=x.device).scatter_(-1, topk_indices, 0)
            # shape (bsz, 1, end_pos)；top-k 位置填 0，其余 -inf
            # 注意：Prefill 用 (bsz,seqlen,seqlen)，Decode 用 (bsz,1,end_pos)，尺寸不同
            scores += index_mask.unsqueeze(2)
            # unsqueeze(2): (bsz,1,end_pos) → (bsz,1,1,end_pos)，广播到 scores (bsz,1,n_heads,end_pos)

            scores = scores.softmax(dim=-1)  # 沿 end_pos 维归一化，shape 不变 (bsz, 1, n_local_heads, end_pos)
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])
            # scores(bsz,1,h,t) @ kv_cache(bsz,t,c) → (bsz, 1, n_local_heads, kv_lora_rank)，在低秩空间聚合
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])
            # wkv_b_v: (n_local_heads, v_head_dim, kv_lora_rank)，从低秩空间投影到 value 空间
            # 输出 (bsz, 1, n_local_heads, v_head_dim)
        x = self.wo(x.flatten(2))
        # flatten(2): (bsz, seqlen, n_local_heads, v_head_dim) → (bsz, seqlen, n_local_heads*v_head_dim)
        # wo 行并行输出：(bsz, seqlen, dim)，内部 all_reduce 聚合各进程结果
        return x  # 返回注意力输出，shape (bsz, seqlen, dim)，dtype 与输入 x 一致


class MLP(nn.Module):
    """
    多层感知机（MLP），使用 SwiGLU 激活函数的前馈网络层。

    结构：SwiGLU(x) = SiLU(w1(x)) ⊙ w3(x)，再通过 w2 降维。
    支持张量并行（w1/w3 列并行，w2 行并行）和延迟 all_reduce 模式（reduce_output=False）。

    属性:
        w1 (nn.Module): SwiGLU 门控分支，列并行线性层，dim → inter_dim。
        w2 (nn.Module): 输出降维，行并行线性层，inter_dim → dim。
        w3 (nn.Module): SwiGLU 值分支，列并行线性层，dim → inter_dim。
    """
    def __init__(self, dim: int, inter_dim: int, reduce_output: bool = True):
        """
        初始化 MLP 层。

        参数:
            dim (int): 输入和输出特征维度（d_model）。
            inter_dim (int): 中间隐藏层维度（升维后的大小）。
            reduce_output (bool): w2 是否立即执行 all_reduce，默认 True；
                                  MoE 的共享专家设为 False 以延迟聚合。
        """
        super().__init__()  # 调用父类初始化
        self.w1 = ColumnParallelLinear(dim, inter_dim)  # SwiGLU门控投影：dim→inter_dim（列并行）
        self.w2 = RowParallelLinear(inter_dim, dim, reduce_output=reduce_output)  # 降维投影：inter_dim→dim（行并行）
        self.w3 = ColumnParallelLinear(dim, inter_dim)  # SwiGLU值投影：dim→inter_dim（列并行）

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MLP 前向传播，应用 SwiGLU 激活函数（门控线性单元）。

        数据流 shape 变化（以 ColumnParallel 切分后为例，part_inter = inter_dim // world_size）：
            x: (..., dim)
                ↓ w1(x): ColumnParallelLinear → (..., part_inter_dim)   # 门控分支
                ↓ w3(x): ColumnParallelLinear → (..., part_inter_dim)   # 值分支
                ↓ SwiGLU = SiLU(w1) ⊙ w3      → (..., part_inter_dim)  # 逐元素门控
                ↓ .type_as(x)                  → (..., part_inter_dim)  # 恢复输入 dtype
                ↓ w2(...): RowParallelLinear   → (..., dim)             # 降维 + all_reduce
            输出: (..., dim)

        参数:
            x (torch.Tensor): 输入张量，形状 (..., dim)，通常为 (batch, seq, dim)。

        返回:
            torch.Tensor: MLP 输出，形状 (..., dim)，dtype 与输入一致。
        """
        # ── SwiGLU 激活计算过程 ───────────────────────────────────────────────────
        # w1(x): (..., dim) → (..., part_inter_dim)，门控分支；ColumnParallelLinear 列切分
        # .float(): 转 float32 防止 SiLU 在 BF16 下精度损失（BF16 数值范围有限）
        # F.silu(...): SiLU(z) = z * sigmoid(z)，平滑非线性激活，shape 不变 (..., part_inter_dim)
        gate = F.silu(self.w1(x).float())  # (..., part_inter_dim)，门控值，控制信息通过量

        # w3(x): (..., dim) → (..., part_inter_dim)，值分支；与 w1 并行计算
        # .float(): 同样转 float32 确保乘法精度一致
        value = self.w3(x).float()  # (..., part_inter_dim)，值向量，携带实际特征信息

        # SwiGLU 门控：gate ⊙ value，逐元素乘法，gate 充当软性开关决定每个特征通道的激活程度
        # .type_as(x): 从 float32 恢复为输入 dtype（BF16 或 FP8），shape 不变 (..., part_inter_dim)
        hidden = (gate * value).type_as(x)  # (..., part_inter_dim)

        # w2: RowParallelLinear，inter_dim → dim（降维）
        # 内部执行 all_reduce（reduce_output=True 时）聚合各进程的列并行结果
        # 最终 shape: (..., dim)，dtype 与 hidden 一致（即与输入 x 一致）
        return self.w2(hidden)  # (..., dim)


class Gate(nn.Module):
    """
    MoE 路由门控模块，决定每个 token 应被路由到哪些专家。

    支持 softmax 和 sigmoid 两种得分函数，以及可选的分组路由策略。
    大模型（dim=7168）还使用可学习偏置进行负载均衡动态修正。

    属性:
        dim (int): 输入特征维度。
        topk (int): 每个 token 激活的路由专家数。
        n_groups (int): 专家分组数，1 表示不分组。
        topk_groups (int): 每次激活的最大专家组数。
        score_func (str): 路由得分函数类型，'softmax' 或 'sigmoid'。
        route_scale (float): 路由权重的全局缩放系数。
        weight (torch.nn.Parameter): 门控权重矩阵，形状 (n_experts, dim)。
        bias (Optional[torch.nn.Parameter]): 负载均衡偏置，仅大模型使用，形状 (n_experts,)。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化路由门控模块。

        参数:
            args (ModelArgs): 模型超参数对象，使用 dim、n_activated_experts、
                              n_expert_groups、n_limited_groups、score_func、
                              route_scale、n_routed_experts 等字段。
        """
        super().__init__()  # 调用父类初始化
        self.dim = args.dim  # 输入特征维度
        self.topk = args.n_activated_experts    # 每个token激活的路由专家数
        self.n_groups = args.n_expert_groups    # 专家分组数（1表示不分组）
        self.topk_groups = args.n_limited_groups  # 每次激活的最大专家组数
        self.score_func = args.score_func  # 路由得分函数：softmax或sigmoid
        self.route_scale = args.route_scale  # 路由权重缩放系数
        self.weight = nn.Parameter(torch.empty(args.n_routed_experts, args.dim))  # 门控权重矩阵，形状(n_experts, dim)
        self.bias = nn.Parameter(
            torch.empty(args.n_routed_experts, dtype=torch.float32)
        ) if self.dim == 7168 else None  # 仅在dim=7168（大模型）时使用偏置（负载均衡修正）

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        路由门控前向传播，计算每个 token 的专家路由权重和索引。

        参数:
            x (torch.Tensor): 输入特征张量，形状 (num_tokens, dim)，float32。

        返回:
            Tuple[torch.Tensor, torch.Tensor]:
                - weights (torch.Tensor): 选中专家的路由权重，形状 (num_tokens, topk)，
                                          经 route_scale 缩放（sigmoid 路由还会归一化）。
                - indices (torch.Tensor): 选中专家的全局索引，形状 (num_tokens, topk)，int64。
        """
        # ── Step 1：计算 token 与所有专家的相似度得分 ──────────────────────────────
        # x: (num_tokens, dim) float32（外部传入时已 .view(-1, dim)）
        # weight: (n_routed_experts, dim) float32
        # linear(x, weight) = x @ weight^T → (num_tokens, n_routed_experts)，float32
        scores = linear(x.float(), self.weight.float())  # (num_tokens, n_routed_experts)

        # ── Step 2：激活函数归一化 ─────────────────────────────────────────────────
        if self.score_func == "softmax":
            # softmax：所有专家得分联合归一化，各 token 的专家得分之和为 1
            # 优点：得分相对竞争，高分专家更突出；适合参数少时使用
            scores = scores.softmax(dim=-1)  # (num_tokens, n_routed_experts)，值域 (0,1)，行和为 1
        else:
            # sigmoid：每个专家得分独立归一化，互不影响
            # 优点：允许多个专家同时高分，更宽松的路由策略
            scores = scores.sigmoid()  # (num_tokens, n_routed_experts)，值域 (0,1)

        # ── 关键设计：保存"干净分数"供最终权重使用 ────────────────────────────────
        # original_scores 记录激活后但加偏置前的得分，用于最终路由权重 gather
        # 原因：偏置仅用于决定"选哪些专家"（影响 topk 选择），不应影响专家权重本身
        original_scores = scores  # (num_tokens, n_routed_experts)，无偏置版本

        if self.bias is not None:
            # ── 负载均衡偏置（仅大模型 dim=7168 使用）────────────────────────────
            # bias: (n_routed_experts,)，广播到 (num_tokens, n_routed_experts)
            # 目的：动态调整各专家被选中的频率，防止负载不均（某些专家总是被选/不被选）
            # 偏置在训练中自动调整，推理时固定，仅影响专家选择，不影响权重
            scores = scores + self.bias  # (num_tokens, n_routed_experts)，加偏置后的选择分数

        if self.n_groups > 1:
            # ── 分组路由策略（n_groups > 1 时启用，DeepSeek MoE 大模型使用）─────────
            # 设计目的：将 n_routed_experts 个专家分为 n_groups 组，
            # 先在组间选 topk_groups 个最优组，再在选中组内选全局 topk 个专家，
            # 减少跨组通信开销（分组与进程分配对应）

            # 分组：将专家得分按组重新排布
            # (num_tokens, n_routed_experts) → (num_tokens, n_groups, experts_per_group)
            # 其中 experts_per_group = n_routed_experts // n_groups
            scores = scores.view(x.size(0), self.n_groups, -1)  # (num_tokens, n_groups, experts_per_group)

            if self.bias is None:
                # 无偏置：取每组内得分最高的专家作为该组的代表得分
                # 直觉：组内最强专家代表该组的"潜力"
                group_scores = scores.amax(dim=-1)  # (num_tokens, n_groups)，每组最大得分
            else:
                # 有偏置：取每组内 top-2 得分之和作为该组代表（DeepSeek 的负载均衡策略）
                # 考虑 top-2 而非 top-1，使组的选择更稳定，避免单个高偏置专家主导
                # topk(2, dim=-1)[0]: (num_tokens, n_groups, 2)，每组前2名得分
                # .sum(-1): 求和得组得分 (num_tokens, n_groups)
                group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)  # (num_tokens, n_groups)

            # 选出得分最高的 topk_groups 个组的索引
            # [1] 取 indices 部分（[0] 是得分值）
            indices = group_scores.topk(self.topk_groups, dim=-1)[1]  # (num_tokens, topk_groups)，dtype int64

            # 构建分组掩码：未被选中的组内所有专家将被屏蔽
            # scores.new_ones: 形状 (num_tokens, n_groups) 的全 True 布尔张量（True=屏蔽）
            # .scatter_(1, indices, False)：将选中组的位置设为 False（不屏蔽），其余保持 True（屏蔽）
            mask = scores.new_ones(x.size(0), self.n_groups, dtype=bool).scatter_(1, indices, False)
            # mask shape: (num_tokens, n_groups)

            # 将未选中组的专家得分填为 -inf（使其无法进入全局 topk），恢复扁平形状
            # mask.unsqueeze(-1): (num_tokens, n_groups) → (num_tokens, n_groups, 1)
            #   广播到 (num_tokens, n_groups, experts_per_group)，屏蔽整组内所有专家
            # .flatten(1): (num_tokens, n_groups, experts_per_group) → (num_tokens, n_routed_experts)
            scores = scores.masked_fill_(mask.unsqueeze(-1), float("-inf")).flatten(1)
            # scores shape: (num_tokens, n_routed_experts)，未选中组的专家位置为 -inf

        # ── Step 3：从（可能已屏蔽的）得分中选出全局 top-k 专家 ─────────────────
        # topk(self.topk, dim=-1)[1]：沿专家维度选 topk 个最高分的索引
        # -inf 的位置（未选中组）不会被选中（softmax/sigmoid 后 -inf 不会赢得 topk）
        indices = scores.topk(self.topk, dim=-1)[1]  # (num_tokens, topk)，int64，全局专家索引

        # ── Step 4：从原始无偏置得分中取出选中专家的权重 ─────────────────────────
        # original_scores: (num_tokens, n_routed_experts)
        # indices: (num_tokens, topk)
        # gather(1, indices)：按 indices 从 original_scores 的专家维（dim=1）取值
        # 输出 shape: (num_tokens, topk)，即选中的 topk 个专家的"干净"路由权重
        weights = original_scores.gather(1, indices)  # (num_tokens, topk)，float32

        if self.score_func == "sigmoid":
            # sigmoid 路由：各专家独立打分，未归一化，需手动归一化使权重和为 1
            # dim=-1 上 keepdim=True 保持 (num_tokens, 1) 广播
            weights /= weights.sum(dim=-1, keepdim=True)  # (num_tokens, topk)，归一化后行和为 1

        # 全局路由权重缩放（标量，控制路由输出的整体幅度）
        weights *= self.route_scale  # (num_tokens, topk)，shape 不变

        return weights, indices
        # weights: (num_tokens, topk)，float32，选中专家的路由权重（经归一化和缩放）
        # indices: (num_tokens, topk)，int64，选中专家的全局编号（0 到 n_routed_experts-1）


class Expert(nn.Module):
    """
    MoE 单个专家模块，结构与 MLP 相同但使用非并行 Linear 层。

    每个专家独立处理路由到该专家的 token 子集，使用 SwiGLU 激活函数。
    与 MLP 不同，Expert 使用普通 Linear（非 ColumnParallelLinear），
    因为专家在进程间是独立分配的，不需要列/行并行切分。

    属性:
        w1 (nn.Module): SwiGLU 门控分支，Linear，dim → inter_dim。
        w2 (nn.Module): 输出降维，Linear，inter_dim → dim。
        w3 (nn.Module): SwiGLU 值分支，Linear，dim → inter_dim。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化专家模块。

        参数:
            dim (int): 输入和输出特征维度（d_model）。
            inter_dim (int): 中间隐藏层维度（MoE 专家的 moe_inter_dim）。
        """
        super().__init__()  # 调用父类初始化
        self.w1 = Linear(dim, inter_dim)   # SwiGLU门控投影（普通Linear，非并行）
        self.w2 = Linear(inter_dim, dim)   # 降维投影
        self.w3 = Linear(dim, inter_dim)   # SwiGLU值投影

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        专家模块前向传播，结构与 MLP 完全相同，使用 SwiGLU 激活。

        与 MLP 的区别：专家使用普通 Linear（非并行），因为专家按进程分配，
        本进程只计算自己负责的专家，无需列/行并行切分。

        数据流 shape 变化（N = num_routed_tokens，即本步路由到该专家的 token 数）：
            x: (N, dim)
                ↓ w1(x): Linear(dim, inter_dim) → (N, inter_dim)   # 门控分支
                ↓ w3(x): Linear(dim, inter_dim) → (N, inter_dim)   # 值分支
                ↓ SwiGLU = SiLU(w1) ⊙ w3        → (N, inter_dim)  # 逐元素门控
                ↓ .type_as(x)                    → (N, inter_dim)  # 恢复 dtype
                ↓ w2(...): Linear(inter_dim, dim) → (N, dim)        # 降维（无 all_reduce）
            输出: (N, dim)

        参数:
            x (torch.Tensor): 路由到本专家的 token 特征，形状 (num_routed_tokens, dim)；
                              num_routed_tokens 随每步 token 分配动态变化（可能为 0）。

        返回:
            torch.Tensor: 专家输出，形状 (num_routed_tokens, dim)，dtype 与输入一致。
        """
        # 门控分支：dim → inter_dim，SiLU 激活（float32 精度）
        gate = F.silu(self.w1(x).float())  # (N, inter_dim)，门控权重

        # 值分支：dim → inter_dim（float32 精度，与 gate 数值精度对齐）
        value = self.w3(x).float()  # (N, inter_dim)，特征值向量

        # SwiGLU 门控并恢复 dtype：gate 决定每维特征通过的"门"，value 提供特征值
        hidden = (gate * value).type_as(x)  # (N, inter_dim)，dtype 与输入 x 一致

        # 降维输出：inter_dim → dim（普通 Linear，无 all_reduce，专家独立计算）
        return self.w2(hidden)  # (N, dim)


class MoE(nn.Module):
    """
    混合专家模块（Mixture-of-Experts，MoE）。

    由路由专家（稀疏激活）和共享专家（密集激活）两部分组成：
    - 路由专家：每个 token 只激活 n_activated_experts 个，减少每步计算量。
    - 共享专家：所有 token 都经过，保证信息不丢失。

    在分布式环境中，路由专家按进程数均匀分配，每个进程只存储和计算本地专家。

    属性:
        dim (int): 输入特征维度。
        n_routed_experts (int): 路由专家总数（全局）。
        n_local_experts (int): 本进程负责的本地路由专家数。
        n_activated_experts (int): 每个 token 激活的路由专家数。
        gate (nn.Module): 路由门控模块，决定 token 的专家分配。
        experts (nn.ModuleList): 全局专家列表，本进程范围外的为 None。
        shared_experts (nn.Module): 共享专家 MLP，所有 token 都经过。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 MoE 模块。

        参数:
            args (ModelArgs): 模型超参数对象，使用 dim、n_routed_experts、
                              n_activated_experts、n_shared_experts、moe_inter_dim 等字段。
        """
        super().__init__()  # 调用父类初始化
        self.dim = args.dim  # 特征维度
        assert args.n_routed_experts % world_size == 0, \
            f"Number of experts must be divisible by world size (world_size={world_size})"  # 断言专家数能被进程数整除
        self.n_routed_experts = args.n_routed_experts  # 路由专家总数
        self.n_local_experts = args.n_routed_experts // world_size  # 本进程负责的本地专家数
        self.n_activated_experts = args.n_activated_experts  # 每token激活的路由专家数
        self.experts_start_idx = rank * self.n_local_experts  # 本进程专家起始全局编号
        self.experts_end_idx = self.experts_start_idx + self.n_local_experts  # 本进程专家结束全局编号（不含）
        self.gate = Gate(args)  # 创建路由门控模块
        self.experts = nn.ModuleList([
            Expert(args.dim, args.moe_inter_dim) if self.experts_start_idx <= i < self.experts_end_idx
            else None  # 本进程范围内创建Expert实例，其他设None节省内存
            for i in range(self.n_routed_experts)
        ])  # 创建全局专家列表，只有本进程范围的是实例
        self.shared_experts = MLP(
            args.dim, args.n_shared_experts * args.moe_inter_dim, reduce_output=False
        )  # 共享专家（所有token都经过），reduce_output=False延迟all_reduce

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MoE 前向传播：路由 token 到对应专家并聚合输出。

        参数:
            x (torch.Tensor): 输入特征，形状 (batch, seq, dim)。

        返回:
            torch.Tensor: MoE 输出（路由专家 + 共享专家之和），
                          形状 (batch, seq, dim)，dtype 与输入一致。
        """
        shape = x.size()  # 保存原始形状(batch, seq, dim)
        x = x.view(-1, self.dim)  # 合并batch和seq维度，形状(-1, dim)
        weights, indices = self.gate(x)  # 计算路由权重和专家索引，各形状(num_tokens, topk)
        y = torch.zeros_like(x, dtype=torch.float32)  # 初始化输出累加器，float32保持精度
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts).tolist()
        # indices.flatten(): (num_tokens*topk,)，展平为 1D；bincount 统计每个专家 id 出现次数
        # counts: list[int]，长度 n_routed_experts，counts[i] 表示路由到专家 i 的 token 数
        for i in range(self.experts_start_idx, self.experts_end_idx):  # 只遍历本进程负责的本地专家（非本进程的为 None）
            if counts[i] == 0:  # 该专家本 step 没有被任何 token 路由到
                continue  # 跳过，避免对空张量执行无意义的前向传播
            expert = self.experts[i]  # 取第 i 个 Expert 实例
            idx, top = torch.where(indices == i)
            # idx: shape (counts[i],)，dtype int64，路由到专家 i 的 token 在 x 中的行号
            # top: shape (counts[i],)，dtype int64，该 token 在其 topk 维度中的位置（用于取对应权重）
            y[idx] += expert(x[idx]) * weights[idx, top, None]
            # x[idx]: (counts[i], dim)，路由到专家 i 的 token 特征
            # expert(x[idx]): (counts[i], dim)，专家前向输出
            # weights[idx, top]: (counts[i],)，对应路由权重；None 扩维为 (counts[i], 1) 以广播
            # y[idx] 按索引 scatter-add 累加路由专家的加权输出
        y += self.shared_experts(x)  # 共享专家输出：(num_tokens, dim)，所有 token 均经过，直接累加
        if world_size > 1:  # 多 GPU：各进程持有不同路由专家的输出，all_reduce 求和得到完整输出
            dist.all_reduce(y)  # y: (num_tokens, dim)，float32，shape 不变
        return y.type_as(x).view(shape)  # 恢复为输入 dtype，view 回 (batch, seq, dim)


class Block(nn.Module):
    """
    Transformer 块，由 MLA 注意力层和前馈网络（MLP 或 MoE）组成。

    采用 Pre-Norm 结构（先归一化再计算）和融合残差连接（将残差累积与 RMSNorm 融合）。
    前 n_dense_layers 层的 FFN 为密集 MLP，之后层的 FFN 为稀疏 MoE。

    属性:
        attn (MLA): 多头潜在注意力层。
        ffn (Union[MLP, MoE]): 前馈网络（密集 MLP 或混合专家 MoE）。
        attn_norm (RMSNorm): 注意力层前的 Pre-Norm 归一化。
        ffn_norm (RMSNorm): FFN 层前的 Pre-Norm 归一化。
    """
    def __init__(self, layer_id: int, args: ModelArgs):
        """
        初始化 Transformer 块。

        参数:
            layer_id (int): 当前层在 Transformer 中的索引（从 0 开始），
                            用于判断使用 MLP（< n_dense_layers）还是 MoE。
            args (ModelArgs): 模型超参数对象。
        """
        super().__init__()  # 调用父类初始化
        self.attn = MLA(args)  # 创建多头潜在注意力层（MLA）
        self.ffn = MLP(args.dim, args.inter_dim) if layer_id < args.n_dense_layers else MoE(args)
        # 前n_dense_layers层使用密集MLP，之后层使用MoE（稀疏激活）
        self.attn_norm = RMSNorm(args.dim)  # 注意力层前的Pre-Norm（RMSNorm）
        self.ffn_norm = RMSNorm(args.dim)   # FFN层前的Pre-Norm（RMSNorm）

    def forward(self, x: torch.Tensor, residual: torch.Tensor, start_pos: int,
                freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        """
        Transformer 块前向传播（Pre-Norm + 融合残差连接）。

        ── 融合残差连接（Fused Residual）概念 ──────────────────────────────────────
        标准 Pre-Norm Transformer 每层执行：
            residual = residual + SubLayer(RMSNorm(residual))
        等价于：先累积残差 x_res = x + attn_out，再归一化 x_norm = RMSNorm(x_res)。

        DeepSeek V3 将这两步融合：
            RMSNorm.forward(x, residual) 内部一步完成：
                new_residual = x + residual          # 累积残差（未归一化，用于继续传递）
                x_norm       = RMSNorm(new_residual) # 归一化结果（送入下一子层）
        优势：减少一次读写 (batch, seq, dim) 大张量，降低显存带宽占用。

        ── 每层数据流 ──────────────────────────────────────────────────────────────
        输入:  x (batch, seq, dim)，residual (batch, seq, dim) 或 None（第一层）

        第一层（residual=None）：
            x_norm   = RMSNorm(x)         # (batch, seq, dim)，归一化供 attn 使用
            residual = x                  # (batch, seq, dim)，保存原始输入作为残差起点

        后续层（residual 已有值）：
            residual = x + residual       # (batch, seq, dim)，累积残差（attn 输出 + 历史残差）
            x_norm   = RMSNorm(residual)  # (batch, seq, dim)，归一化供 attn 使用

        注意力子层：
            attn_out = MLA(x_norm, ...)   # (batch, seq, dim)
            x        = attn_out           # 注意力输出作为下一步 x，传入 ffn_norm 融合

        FFN 前 Pre-Norm（同样融合）：
            residual = x + residual       # (batch, seq, dim)，注意力输出累积到残差
            x_norm   = RMSNorm(residual)  # (batch, seq, dim)，供 FFN 使用

        FFN：
            ffn_out = MLP/MoE(x_norm)     # (batch, seq, dim)

        输出:  ffn_out (batch, seq, dim)，residual (batch, seq, dim) 传给下一层

        参数:
            x (torch.Tensor): 上一层/步的子模块输出（注意力输出或 FFN 输出），
                              形状 (batch, seq, dim)。第一层时为词嵌入输出。
            residual (torch.Tensor | None): 跨层累积残差张量，形状 (batch, seq, dim)；
                                             第一层时为 None（尚无历史残差）。
            start_pos (int): KV 缓存起始位置（prefill=0，decode=已处理 token 数）。
            freqs_cis (torch.Tensor): 预计算的 RoPE 复数因子，形状 (seqlen, rope_dim//2)。
            mask (Optional[torch.Tensor]): 因果注意力掩码，形状 (seqlen, seqlen)；
                                           decode 单步时为 None。

        返回:
            Tuple[torch.Tensor, torch.Tensor]:
                - x       (batch, seq, dim)：当前层 FFN 输出，即"注意力输出 x"，
                                             传给下一层作为 x 输入（而非 residual）。
                - residual (batch, seq, dim)：更新后的残差累积（已融合本层注意力输出），
                                              传给下一层在 FFN 前进一步融合。
        """
        if residual is None:
            # ── 第一层特殊处理：无历史残差，直接归一化 ──────────────────────────
            # x 既作为 residual 的起点，也被归一化后送入注意力
            # attn_norm(x): RMSNorm，输出 shape (batch, seq, dim)
            # 原始 x 保存为 residual，供后续层在 attn_norm(x, residual) 中累积
            x, residual = self.attn_norm(x), x
            # 此后：x = RMSNorm(原始输入)，residual = 原始输入
        else:
            # ── 后续层：融合残差后归一化（一步完成累积+归一化）───────────────────
            # attn_norm(x, residual) 内部：
            #   new_residual = x + residual   # 注意力输出与历史残差相加，shape (batch, seq, dim)
            #   x_norm = RMSNorm(new_residual) # 归一化，shape (batch, seq, dim)
            # 返回 (x_norm, new_residual)，两者 shape 均为 (batch, seq, dim)
            x, residual = self.attn_norm(x, residual)
            # 此后：x = RMSNorm(attn_out + old_residual)，residual = attn_out + old_residual（未归一化）

        # ── 注意力子层 ──────────────────────────────────────────────────────────
        # 输入 x: 经过 Pre-Norm 的归一化特征，(batch, seq, dim)
        # 输出 x: MLA 注意力输出，(batch, seq, dim)
        x = self.attn(x, start_pos, freqs_cis, mask)  # (batch, seq, dim)

        # ── FFN 前 Pre-Norm（同样融合残差）──────────────────────────────────────
        # 输入：x=注意力输出 (batch,seq,dim)，residual=attn_norm 后的残差 (batch,seq,dim)
        # 内部：new_residual = attn_out + residual；x_norm = RMSNorm(new_residual)
        # 输出：(x_norm, new_residual)，两者 shape 均为 (batch, seq, dim)
        x, residual = self.ffn_norm(x, residual)  # (batch, seq, dim), (batch, seq, dim)

        # ── 前馈网络子层（MLP 或 MoE）────────────────────────────────────────────
        # 输入 x: 经过 ffn_norm 归一化的特征，(batch, seq, dim)
        # 输出 x: FFN 变换结果，(batch, seq, dim)，携带本层的前馈特征表示
        x = self.ffn(x)  # (batch, seq, dim)

        return x, residual
        # x:        (batch, seq, dim)，本层 FFN 输出，传给下一层作为 x（将在下一层 attn_norm 中与 residual 融合）
        # residual: (batch, seq, dim)，包含 attn_out + 历史残差的未归一化累积，传给下一层 attn_norm 融合


class Transformer(nn.Module):
    """
    完整的 DeepSeek V3.2 Transformer 模型。

    由词嵌入层、多个 Transformer Block 堆叠、最终归一化和 LM Head 组成。
    支持张量并行推理（词嵌入和 LM Head 列并行，注意力/MLP 混合并行）。

    属性:
        max_seq_len (int): 最大序列长度（来自 args.max_seq_len）。
        embed (ParallelEmbedding): 支持张量并行的词嵌入层。
        layers (torch.nn.ModuleList): Transformer Block 列表，共 n_layers 层。
        norm (RMSNorm): 所有 Block 之后的最终归一化层。
        head (ColumnParallelLinear): LM Head，将 dim 维特征映射到词汇表大小（float32）。
        freqs_cis (torch.Tensor): 预计算的 RoPE 复数因子（buffer），
                                  形状 (max_seq_len, qk_rope_head_dim//2)。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化完整 Transformer 模型，并设置全局分布式状态和权重精度。

        参数:
            args (ModelArgs): 模型超参数对象，包含所有模型配置。
        """
        global world_size, rank  # 声明修改全局变量
        world_size = dist.get_world_size() if dist.is_initialized() else 1  # 从分布式状态获取总进程数（未初始化则1）
        rank = dist.get_rank() if dist.is_initialized() else 0  # 从分布式状态获取当前进程编号（未初始化则0）
        Linear.dtype = torch.float8_e4m3fn if args.dtype == "fp8" else torch.bfloat16  # 根据配置设置全局权重类型
        Linear.scale_fmt = args.scale_fmt  # 设置全局FP8缩放因子格式
        super().__init__()  # 调用父类nn.Module初始化
        self.max_seq_len = args.max_seq_len  # 保存最大序列长度
        self.embed = ParallelEmbedding(args.vocab_size, args.dim)  # 创建支持张量并行的词嵌入层
        self.layers = torch.nn.ModuleList()  # 创建空的Transformer层列表
        for layer_id in range(args.n_layers):  # 循环创建所有Transformer层
            self.layers.append(Block(layer_id, args))  # 创建第layer_id层并添加（前几层MLP，后续MoE）
        self.norm = RMSNorm(args.dim)  # 最终的RMSNorm（在LM head之前）
        # lm_head存储为bf16，这里用fp32便于精确计算logits
        self.head = ColumnParallelLinear(args.dim, args.vocab_size, dtype=torch.float32)  # LM输出头：dim→vocab_size（列并行，float32）
        self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)
        # 预计算RoPE复数因子并注册为buffer，形状(max_seq_len, rope_dim//2)，不保存到checkpoint

    @torch.inference_mode()
    def forward(self, tokens: torch.Tensor, start_pos: int = 0):
        """
        Transformer 完整前向传播（推理模式，不计算梯度）。

        参数:
            tokens (torch.Tensor): 输入 token ID 张量，形状 (batch_size, seq_len)，int64。
            start_pos (int): 当前序列在完整上下文中的起始位置，用于 KV 缓存和 RoPE 索引，
                             prefill 时为 0，decode 时为已处理的 token 数量。

        返回:
            torch.Tensor: 最后一个 token 位置的 logits，形状 (batch_size, vocab_size)，float32。
        """
        seqlen = tokens.size(1)  # int：当前输入序列长度（tokens shape: (batch_size, seqlen)，int64）
        freqs_cis = self.freqs_cis[start_pos:start_pos + seqlen]
        # 从预计算缓存中切片当前位置对应的 RoPE 因子，shape (seqlen, qk_rope_head_dim//2)，complex64
        mask = torch.full(
            (seqlen, seqlen), float("-inf"), device=tokens.device
        ).triu_(1) if seqlen > 1 else None
        # Prefill（seqlen>1）：triu_(1) 将主对角线以上填 -inf，得到上三角 causal mask，shape (seqlen, seqlen)
        # Decode（seqlen=1）：单步无需 mask，置 None
        h, residual = self.embed(tokens), None
        # h: 词嵌入输出，shape (batch_size, seqlen, dim)；residual 初始为 None（第一层 Block 特殊处理）
        for layer in self.layers:  # 逐层通过 n_layers 个 Transformer Block
            h, residual = layer(h, residual, start_pos, freqs_cis, mask)
            # 每层输入/输出：h 和 residual shape 均为 (batch_size, seqlen, dim)
        h, _ = self.norm(h, residual)
        # 最终 RMSNorm：融合最后一层残差 (h + residual) 后归一化，h shape (batch_size, seqlen, dim)
        # _ 为更新后的 residual（已不再使用，丢弃）
        logits = self.head(h[:, -1].float())
        # h[:, -1]: 取最后一个 token 位置的特征，shape (batch_size, dim)
        # .float(): 转 float32 确保 logits 精度（LM head 权重已是 float32）
        # head 为列并行，输出 (batch_size, part_vocab_size)，其中 part_vocab_size = vocab_size // world_size
        if world_size > 1:  # 多 GPU：每个进程只有词汇表的一个分片，需聚合
            all_logits = [torch.empty_like(logits) for _ in range(world_size)]
            # 为每个进程创建 shape (batch_size, part_vocab_size) 的接收缓冲区
            dist.all_gather(all_logits, logits)  # 将各进程的 logits 分片写入 all_logits 列表
            logits = torch.cat(all_logits, dim=-1)  # 沿词汇表维拼接，shape (batch_size, vocab_size)
        return logits  # 返回完整 logits，shape (batch_size, vocab_size)，dtype float32


if __name__ == "__main__":
    torch.set_default_dtype(torch.bfloat16)     # 设置默认张量类型为BF16
    torch.set_default_device("cuda")             # 设置默认设备为CUDA
    torch.manual_seed(0)                         # 固定随机种子
    args = ModelArgs()                           # 使用默认参数创建ModelArgs实例
    x = torch.randint(0, args.vocab_size, (2, 128))  # 生成随机输入token（batch_size=2, seq_len=128）
    model = Transformer(args)                    # 创建完整Transformer模型
    print(model(x).size())                       # 打印输出logits形状（应为torch.Size([2, vocab_size])）